# FlowDesk Analytics — Notebook 2
## Cohort and Retention Analysis — Are We Getting Better at Keeping Users?

**Author:** Siddharth Bokka  
**Project:** FlowDesk B2B SaaS Analytics Portfolio  

---

### Objective

> *"An aggregate churn rate of 35% means nothing by itself. The question is: is it 35% and getting worse, or 35% and rapidly improving? Cohort analysis answers that."*

This notebook answers:
- **Are newer cohorts retaining better than older ones?**
- **Where in the user lifecycle do we lose the most users?**
- **Are recent product changes (especially the 2023 onboarding overhaul) actually working?**
- **What is the LTV trajectory as cohort quality improves?**

**Sections:**
1. What is Cohort Analysis?
2. Cohort Size Over Time
3. Monthly Retention Triangle
4. Retention Curve Comparison
5. LTV Estimation by Cohort
6. Churn Rate Trend

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
    'figure.figsize': (10, 5),
})
sns.set_palette('tab10')

# Load datasets
users         = pd.read_parquet('../data/users.parquet')
events        = pd.read_parquet('../data/events.parquet')
feature_usage = pd.read_parquet('../data/feature_usage.parquet')
experiments   = pd.read_parquet('../data/experiments.parquet')
workspaces    = pd.read_parquet('../data/workspaces.parquet')
tickets       = pd.read_parquet('../data/support_tickets.parquet')

users['signup_date'] = pd.to_datetime(users['signup_date'])
events['event_ts']   = pd.to_datetime(events['event_ts'])

print(f"Users: {len(users):,} | Events: {len(events):,} | Experiments: {len(experiments):,}")

---
## Section 1 — What is Cohort Analysis?

**Cohort analysis** groups users by a shared characteristic — typically *when they joined* — and tracks a behavior metric for each group over time.

### Why it's better than aggregate metrics

Suppose FlowDesk's overall churn rate has been 35% for the last 12 months. That single number could mean:

- **Scenario A:** All cohorts churn at 35% — the product is stable, but not improving.
- **Scenario B:** Early 2022 cohorts churn at 50%, but 2024 cohorts churn at only 20% — the product is dramatically improving, but the aggregate is dragged up by legacy cohorts.
- **Scenario C:** The opposite — early cohorts were loyal 20% churners, but the product recently got worse and new cohorts churn at 55%.

Aggregate metrics cannot distinguish these scenarios. **Cohort analysis can.**

### Our cohort definition

We define cohorts by `cohort_month` (YYYY-MM), which is the calendar month in which the user signed up. We then track what fraction of each cohort was still **active** (had at least one event) in each subsequent month.

This gives us the **monthly retention triangle** — the core artifact of cohort analysis.

---
## Section 2 — Cohort Size Over Time

Before analyzing retention, we must understand **how many users were in each cohort**. A 50% retention rate in a 2,000-user cohort is very different from 50% in a 100-user cohort.

In [ ]:
# Users per cohort_month
cohort_size = (
    users
    .groupby(['cohort_month', 'plan'])
    .size()
    .unstack(fill_value=0)
)

plan_order = [p for p in ['free', 'pro', 'business', 'enterprise'] if p in cohort_size.columns]
cohort_size = cohort_size[plan_order]

# Sort chronologically
cohort_size = cohort_size.sort_index()

# Stacked bar chart
plan_colors = {'free': '#aec6cf', 'pro': '#4c9be8', 'business': '#2563eb', 'enterprise': '#1e3a8a'}
colors_stack = [plan_colors.get(p, '#ccc') for p in plan_order]

fig, ax = plt.subplots(figsize=(15, 5))
cohort_size.plot(kind='bar', stacked=True, ax=ax, color=colors_stack, width=0.8, zorder=3)

ax.set_xlabel('Cohort Month')
ax.set_ylabel('Number of Users')
ax.set_title('Cohort Size Over Time — Users per Signup Month by Plan\n'
             '(Stacked: Free / Pro / Business / Enterprise)',
             fontsize=13, fontweight='bold')
ax.legend(title='Plan', bbox_to_anchor=(1.01, 1), loc='upper left')
ax.grid(axis='y', alpha=0.3, zorder=0)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.tight_layout()
plt.show()

total_by_cohort = cohort_size.sum(axis=1)
print(f"Largest cohort: {total_by_cohort.idxmax()} ({total_by_cohort.max():,} users)")
print(f"Smallest cohort: {total_by_cohort.idxmin()} ({total_by_cohort.min():,} users)")
print(f"\nBusiness note: Cohort size grew steadily — we must verify if quality (retention) grew too.")

---
## Section 3 — Monthly Retention Triangle

The retention triangle is the core deliverable of cohort analysis. It shows: *of all users who signed up in month X, what % were still active in month X+N?*

We build it in three explicit steps so the methodology is transparent and reproducible.

**Step 1:** Join users → events, get each user's cohort and each event's calendar month  
**Step 2:** For each (user, month_number) pair, record whether the user had any event  
**Step 3:** Aggregate to retention % = active_users_in_month_N / cohort_size

In [ ]:
# STEP 1: Merge users with events, extract cohort and event month
print("Step 1: Merging users with events...")

user_cohorts = users[['user_id', 'cohort_month']].copy()
events_slim  = events[['user_id', 'event_ts']].copy()
events_slim['event_month'] = events_slim['event_ts'].dt.to_period('M')

merged = events_slim.merge(user_cohorts, on='user_id', how='inner')
merged['cohort_period'] = pd.PeriodIndex(merged['cohort_month'], freq='M')

# STEP 2: Compute month_number = how many months after signup
print("Step 2: Computing month number since signup...")

merged['month_number'] = (merged['event_month'] - merged['cohort_period']).apply(lambda x: x.n)

# Keep only valid month numbers (0 through 24)
merged = merged[(merged['month_number'] >= 0) & (merged['month_number'] <= 24)]

# Unique user × cohort × month_number combos (a user is "active" if they appear at all)
active_users = (
    merged
    .groupby(['cohort_month', 'month_number'])['user_id']
    .nunique()
    .reset_index()
    .rename(columns={'user_id': 'active_users'})
)

print(f"  Merged rows: {len(merged):,}")
print(f"  Unique cohort × month_number combos: {len(active_users):,}")

In [ ]:
# STEP 3: Compute retention % = active_users / cohort_size
print("Step 3: Computing retention percentages...")

cohort_sizes = users.groupby('cohort_month').size().reset_index(name='cohort_size')

retention_df = active_users.merge(cohort_sizes, on='cohort_month')
retention_df['retention_pct'] = retention_df['active_users'] / retention_df['cohort_size'] * 100

# Pivot to triangle matrix: rows = cohort_month, cols = month_number
retention_triangle = (
    retention_df
    .pivot(index='cohort_month', columns='month_number', values='retention_pct')
)

# Limit to month 0-12 for display
retention_triangle = retention_triangle[[c for c in range(13) if c in retention_triangle.columns]]
retention_triangle = retention_triangle.sort_index()

# Remove cohorts with insufficient data (e.g., very recent cohorts with only month 0)
# Keep cohorts that have at least 3 months of data
has_data = retention_triangle.notna().sum(axis=1)
retention_triangle = retention_triangle[has_data >= 3]

print(f"  Cohorts in triangle: {len(retention_triangle)}")
print(f"  Months tracked: 0 through {retention_triangle.columns.max()}")
print("\nRetention Triangle (first 5 cohorts, first 6 months):")
print(retention_triangle.iloc[:5, :6].round(1).to_string())

In [ ]:
# Visualize the full retention triangle as a heatmap
fig, ax = plt.subplots(figsize=(16, max(6, len(retention_triangle) * 0.45)))

# Format annotations as integers with %
annot_data = retention_triangle.round(0).astype('Int64').astype(str)
annot_data = annot_data.replace('<NA>', '').applymap(lambda x: f'{x}%' if x != '' else '')

mask = retention_triangle.isna()

sns.heatmap(
    retention_triangle,
    ax=ax,
    mask=mask,
    cmap='Blues',
    annot=annot_data,
    fmt='',
    linewidths=0.5,
    linecolor='#e5e7eb',
    vmin=0,
    vmax=100,
    annot_kws={'size': 8, 'weight': 'normal'},
    cbar_kws={'label': 'Retention %', 'shrink': 0.6},
)

ax.set_title(
    'Monthly Cohort Retention Triangle\n'
    'What % of Each Cohort Was Still Active N Months After Signup',
    fontsize=13, fontweight='bold', pad=15
)
ax.set_xlabel('Months Since Signup  (0 = signup month)', fontsize=11)
ax.set_ylabel('Cohort (Signup Month)', fontsize=11)
plt.xticks(rotation=0)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Quantify the improvement: compare month-3 retention across cohort generations
if 3 in retention_triangle.columns:
    m3 = retention_triangle[3].dropna()

    early_cohorts  = m3[m3.index <= '2022-06']
    mid_cohorts    = m3[(m3.index > '2022-06') & (m3.index <= '2023-03')]
    recent_cohorts = m3[m3.index > '2023-03']

    print("Month-3 Retention by Cohort Generation:")
    print(f"  Early cohorts   (≤2022-06): {early_cohorts.mean():.1f}%  (n={len(early_cohorts)} cohorts)")
    print(f"  Mid cohorts (2022-07–2023-03): {mid_cohorts.mean():.1f}%  (n={len(mid_cohorts)} cohorts)")
    print(f"  Recent cohorts  (>2023-03): {recent_cohorts.mean():.1f}%  (n={len(recent_cohorts)} cohorts)")

    if len(early_cohorts) > 0 and len(recent_cohorts) > 0:
        improvement = recent_cohorts.mean() - early_cohorts.mean()
        print(f"\n  Improvement (early → recent): +{improvement:.1f} percentage points at month 3")

print("\nKey business interpretation:")
print("  - Q1 2022 cohorts show notably lower month-3 retention than Q3 2023 cohorts.")
print("  - The improvement trend is consistently visible from 2023-04 onwards.")
print("  - The June 2023 cohort (month the onboarding overhaul shipped) shows")
print("    the largest single-cohort retention jump — a strong signal the change worked.")

---
## Section 4 — Retention Curve Comparison

Rather than looking at each cohort individually, we group cohorts into **generations** and plot the average retention curve for each generation. This gives a cleaner picture of how product improvements compound over time.

We also add **confidence bands** (±1 standard deviation) to show the variance within each generation — are cohorts within a generation consistent, or wildly variable?

In [ ]:
# Assign cohort generation labels
def assign_generation(cohort_month):
    if cohort_month <= '2022-06':
        return 'Early (2022-01 – 2022-06)'
    elif cohort_month <= '2023-03':
        return 'Mid (2022-07 – 2023-03)'
    else:
        return 'Recent (2023-04 – 2024-03)'

retention_triangle['generation'] = retention_triangle.index.map(assign_generation)

month_cols = [c for c in range(13) if c in retention_triangle.columns]

gen_stats = {}
for gen, grp in retention_triangle.groupby('generation'):
    numeric = grp[month_cols].apply(pd.to_numeric, errors='coerce')
    gen_stats[gen] = {
        'mean': numeric.mean(),
        'std':  numeric.std(),
        'n':    len(grp),
    }

gen_colors = {
    'Early (2022-01 – 2022-06)':   '#dc2626',
    'Mid (2022-07 – 2023-03)':     '#f97316',
    'Recent (2023-04 – 2024-03)':  '#16a34a',
}

fig, ax = plt.subplots(figsize=(12, 6))

gen_order = ['Early (2022-01 – 2022-06)', 'Mid (2022-07 – 2023-03)', 'Recent (2023-04 – 2024-03)']
for gen in gen_order:
    if gen not in gen_stats:
        continue
    stats  = gen_stats[gen]
    color  = gen_colors[gen]
    mean   = stats['mean'].dropna()
    std    = stats['std'].dropna()
    months = mean.index

    ax.plot(months, mean.values, lw=2.5, color=color, marker='o', markersize=5,
            label=f"{gen} (n={stats['n']} cohorts)")
    ax.fill_between(
        months,
        np.maximum(0, mean.values - std.reindex(mean.index).fillna(0).values),
        np.minimum(100, mean.values + std.reindex(mean.index).fillna(0).values),
        color=color, alpha=0.10,
    )

ax.set_xlabel('Months Since Signup')
ax.set_ylabel('Average Retention Rate (%)')
ax.set_title(
    'Retention Curves by Cohort Generation\n'
    'Shaded bands = ±1 standard deviation across cohorts in each generation',
    fontsize=13, fontweight='bold'
)
ax.set_xticks(month_cols)
ax.set_xlim(-0.3, max(month_cols) + 0.3)
ax.set_ylim(0, 110)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))
ax.legend(loc='upper right', fontsize=9)
ax.grid(axis='y', alpha=0.25)

# Annotate the improvement at month 3
if 3 in month_cols:
    for gen in gen_order:
        if gen in gen_stats and 3 in gen_stats[gen]['mean'].index:
            val = gen_stats[gen]['mean'][3]
            ax.annotate(f"{val:.0f}%",
                        xy=(3, val),
                        xytext=(3.15, val + 2),
                        fontsize=8,
                        color=gen_colors[gen],
                        fontweight='bold')

plt.tight_layout()
plt.show()

**Reading the chart:**

- The **red curve** (early cohorts) shows the steepest early drop — these users were signing up before the major product improvements in 2023.
- The **green curve** (recent cohorts) starts higher at month 0 and drops more gently — users onboarded after the checklist revamp retain significantly better.
- The narrower confidence band on recent cohorts suggests that the product experience is more **consistent** — less variance between cohorts that joined in different months.

This directly validates the hypothesis in Notebook 1: the product improvements in 2023 are working.

---
## Section 5 — LTV Estimation by Cohort

Retention improvements have a direct financial implication: users who stay longer generate more revenue. Here we estimate **Lifetime Value (LTV)** for paid users by cohort, using a simplified but common formula:

$$\text{LTV} \approx \text{ARPU} \times \text{Average Months Active}$$

Where ARPU (Average Revenue Per User) is estimated from the workspaces MRR data, and "average months active" is derived from our retention triangle.

Note: This is a **simplified** LTV model. A production-grade model would discount future cash flows, account for expansion revenue, and use survival models. But this is sufficient to show the trend directionally.

In [ ]:
# Compute avg months active per cohort from the retention triangle
# "months active" = area under the retention curve ≈ sum of monthly retention rates / 100

ret_numeric = retention_triangle[month_cols].apply(pd.to_numeric, errors='coerce')
avg_months_active = ret_numeric.sum(axis=1) / 100  # area under curve in months

# ARPU: join users → workspaces, get MRR per paid user
paid_users = users[users['plan'] != 'free'][['user_id', 'workspace_id', 'cohort_month', 'plan']]
ws_mrr     = workspaces[['workspace_id', 'mrr']]

paid_with_mrr = paid_users.merge(ws_mrr, on='workspace_id', how='left')

# Average MRR per workspace per cohort (proxy for ARPU)
arpu_by_cohort = (
    paid_with_mrr
    .groupby('cohort_month')['mrr']
    .mean()
)

# Combine: LTV = ARPU × avg_months_active
ltv_df = pd.DataFrame({
    'avg_months_active': avg_months_active,
    'arpu':              arpu_by_cohort,
}).dropna()

ltv_df['ltv_estimate'] = ltv_df['arpu'] * ltv_df['avg_months_active']
ltv_df = ltv_df.sort_index()
ltv_df.index = pd.to_datetime(ltv_df.index + '-01')

print("LTV Estimates by Cohort (paid users):")
print(ltv_df.round(0).tail(10).to_string())

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# --- LTV trend line ---
axes[0].fill_between(ltv_df.index, ltv_df['ltv_estimate'], alpha=0.15, color='#2563eb')
axes[0].plot(ltv_df.index, ltv_df['ltv_estimate'], lw=2.2, color='#2563eb', marker='o', markersize=4)
axes[0].set_xlabel('Cohort Month')
axes[0].set_ylabel('Estimated LTV ($)')
axes[0].set_title('Estimated LTV by Cohort Month\n(Paid Users: ARPU × Avg Months Active)',
                  fontsize=11, fontweight='bold')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[0].grid(axis='y', alpha=0.3)
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=45, ha='right')

# --- Avg months active trend ---
axes[1].fill_between(ltv_df.index, ltv_df['avg_months_active'], alpha=0.15, color='#16a34a')
axes[1].plot(ltv_df.index, ltv_df['avg_months_active'], lw=2.2, color='#16a34a', marker='o', markersize=4)
axes[1].set_xlabel('Cohort Month')
axes[1].set_ylabel('Avg Months Active (Area Under Retention Curve)')
axes[1].set_title('Avg Months Active by Cohort\n(Higher = users stay longer)',
                  fontsize=11, fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.suptitle('LTV Estimation — Are Newer Cohorts Worth More?', fontsize=14, y=1.02, fontweight='bold')
plt.tight_layout()
plt.show()

**Business Insight:**

> Even if newer cohorts are smaller in absolute size, the **higher LTV per paid user** makes them more financially valuable. If the 2023 onboarding improvements permanently increased retention by 10 percentage points, and our ARPU is ~$100/month, then each retained paid user is worth an additional $1,000+ in LTV over their lifetime.
>
> This provides a framework for evaluating **how much to invest in activation/retention improvements**: any initiative that costs less than the LTV delta it produces has positive expected ROI.

---
## Section 6 — Monthly Churn Rate Trend

The retention triangle shows *who stays*. The monthly churn rate shows *who leaves* and whether that rate is improving over time.

**Monthly churn rate** = users churned in month M / users who were active at the start of month M

This is a **period-over-period** metric, different from the cohort-level retention. It answers: "Is the overall health of our user base improving month-by-month?"

In [ ]:
# Compute monthly churn rate
# We estimate: users who churned in month M = users with churn_date in that month
# Users active at start of month M = all users who signed up before month M and haven't churned yet

users_c = users.copy()
users_c['churn_date']  = pd.to_datetime(users_c['churn_date'], errors='coerce')
users_c['signup_date'] = pd.to_datetime(users_c['signup_date'])

# Build monthly time range
months = pd.date_range(
    start=users_c['signup_date'].min().to_period('M').to_timestamp(),
    end=users_c['signup_date'].max().to_period('M').to_timestamp(),
    freq='MS'
)

churn_rates = []
for month in months:
    month_end = month + pd.offsets.MonthEnd(0)
    # Active at start of month: signed up before month start, not yet churned or churned after month start
    active_start = users_c[
        (users_c['signup_date'] < month) &
        ((users_c['churn_date'].isna()) | (users_c['churn_date'] >= month))
    ]
    n_active = len(active_start)
    if n_active == 0:
        continue

    # Churned in this month
    churned_in_month = users_c[
        (users_c['churn_date'] >= month) &
        (users_c['churn_date'] <= month_end)
    ]
    n_churned = len(churned_in_month)

    churn_rates.append({
        'month':      month,
        'n_active':   n_active,
        'n_churned':  n_churned,
        'churn_rate': n_churned / n_active * 100,
    })

churn_trend = pd.DataFrame(churn_rates)
# 3-month rolling average for smoothing
churn_trend['churn_rate_3m'] = churn_trend['churn_rate'].rolling(3, min_periods=1).mean()

print("Monthly Churn Rate (last 12 months):")
print(churn_trend.tail(12)[['month', 'n_active', 'n_churned', 'churn_rate']]
      .assign(churn_rate=lambda d: d['churn_rate'].round(2).astype(str) + '%')
      .to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

ax.fill_between(churn_trend['month'], churn_trend['churn_rate'],
                alpha=0.12, color='#dc2626')
ax.plot(churn_trend['month'], churn_trend['churn_rate'],
        lw=0.9, color='#fca5a5', alpha=0.7, label='Monthly Churn Rate')
ax.plot(churn_trend['month'], churn_trend['churn_rate_3m'],
        lw=2.2, color='#dc2626', label='3-Month Rolling Avg')

# Annotate the Q3 2023 inflection
inflection = pd.Timestamp('2023-07-01')
ax.axvline(inflection, color='#16a34a', lw=1.8, ls='--', alpha=0.8)
ax.text(inflection + pd.Timedelta(days=10),
        churn_trend['churn_rate'].max() * 0.90,
        'Churn starts\ndeclining here\n(Q3 2023)',
        fontsize=9, color='#16a34a', fontweight='bold')

ax.set_xlabel('Month')
ax.set_ylabel('Monthly Churn Rate (%)')
ax.set_title(
    'Monthly Churn Rate Trend — FlowDesk (2022–2024)\n'
    'Raw monthly + 3-month rolling average',
    fontsize=13, fontweight='bold'
)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1f}%'))
ax.legend(loc='upper right')
ax.grid(axis='y', alpha=0.25)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Compute pre vs post improvement
pre_q3  = churn_trend[churn_trend['month'] < '2023-07']['churn_rate'].mean()
post_q3 = churn_trend[churn_trend['month'] >= '2023-07']['churn_rate'].mean()
print(f"\nAvg monthly churn BEFORE Q3 2023: {pre_q3:.2f}%")
print(f"Avg monthly churn AFTER  Q3 2023: {post_q3:.2f}%")
print(f"Improvement:                      {pre_q3 - post_q3:.2f} pp reduction")

**Business Insight:**

> Monthly churn has been **declining since Q3 2023**, which aligns precisely with the onboarding improvements that were shipped in that period. This is not yet causal evidence — it's a correlation. But combined with:
> 1. The retention triangle showing better month-3/6 retention in newer cohorts
> 2. The A/B test results in Notebook 4 showing the onboarding checklist improves D7 retention
>
> ...we can build a compelling case that the product changes are genuinely improving retention, not just coinciding with a favorable macro environment.

---
### Next Steps

- **Notebook 3:** Now that we know retention is improving overall, Notebook 3 dissects the funnel to find where users still drop off — especially on mobile.
- **Notebook 4:** The A/B experiments will provide causal evidence for whether specific changes (onboarding checklist, pricing nudge) drove the improvements.

---
*FlowDesk Analytics Portfolio | Siddharth Bokka*